# Workspace Management

Create a workspace, upload a catalogue and preview images, then share the images.

Use the [demo accounts](getting-started/quickstart.md#environment-assumptions), or change the service URLs and OAuth settings for your deployment.

Adapted from the EOEPCA demo [Workspace Management notebook](https://github.com/EOEPCA/demo/blob/main/demoroot/notebooks/05%20Workspace%20Management.ipynb).


## Run the notebook

From the repository root:

```bash
uv sync
uv run jupyter lab docs/workspace-management.ipynb
```

Keep [notebook_support.py](notebook_support.py) beside the notebook. The service URLs and OAuth settings are in `eoepca-demo.env` at the repository root. Settings come from that file rather than environment variables.

Put `catalog.v1.parquet` in the notebook's working directory before running the cells. It must contain STAC Items with `rendered_preview` or `thumbnail` assets.


In [1]:
import datetime
import re
import jwt
from notebook_support import (
    ApiSession, load_config, print_config,
    token_via_device, wait_for_workspace,
)
api = ApiSession()

The public OAuth client `workspace-api` needs S256 PKCE and Device Authorization Grant enabled. If `CLIENT_ID` names another client, set `WORKSPACE_CLIENT_ID=workspace-api` in the settings file.

Load the settings and check the service addresses:


In [2]:
config = load_config("eoepca-demo.env", "workspace")
print_config(config)

workspace_api_endpoint = config["WORKSPACE_API_ENDPOINT"]

{
  "ENVIRONMENT": "eoepca-demo",
  "OIDC_ISSUER": "https://iam-auth.develop.eoepca.org/realms/eoepca",
  "EOAPI_CLIENT_ID": "",
  "WORKSPACE_CLIENT_ID": "workspace-api",
  "OIDC_SCOPE": "roles offline_access",
  "STAC_ENDPOINT": "",
  "WORKSPACE_API_ENDPOINT": "https://workspace-api.develop.eoepca.org/workspaces"
}
OIDC discovery: https://iam-auth.develop.eoepca.org/realms/eoepca/.well-known/openid-configuration


In [3]:
from pathlib import Path

import pyarrow.parquet as pq
import requests

catalogue = Path("catalog.v1.parquet")
items = pq.read_table(catalogue).to_pylist()
if not items:
    raise ValueError("The catalogue contains no STAC Items.")
print(f"Catalogue ready: {catalogue.name} ({len(items)} items)")


Catalogue ready: catalog.v1.parquet (3 items)


Enter `alice`, `bob`, or `eric` as the owner and complete device login as that user.

If the owner cannot create workspaces, sign in as `oscar` when prompted. Use a fresh private browser window for each account. See [roles](design/iam-integration.md#roles) for the required permissions.


In [5]:
def login_workspace_as(username):
    print(f"Sign in as {username} in a fresh private browser window.")
    tokens = token_via_device(config, "workspace")
    access_token = tokens["access_token"]
    # These claims select the notebook flow. The API validates the token.
    claims = jwt.decode(access_token, options={"verify_signature": False})
    if claims.get("preferred_username") != username:
        raise ValueError(f"Wrong account. Start login again and sign in as {username}.")
    is_admin = "admin" in claims.get("resource_access", {}).get("workspace-api", {}).get("roles", [])
    print(f"Signed in as {username}; workspace administrator: {is_admin}")
    return access_token, is_admin

owner = input("Workspace owner (alice, bob, or eric) [alice]: ").strip() or "alice"
if not owner:
    raise ValueError("Enter the workspace owner's username.")
token = None
owner_token, owner_is_admin = login_workspace_as(owner)

Workspace owner (alice, bob, or eric) [alice]:  


Sign in as alice in a fresh private browser window.


SSLError: HTTPSConnectionPool(host='iam-auth.develop.eoepca.org', port=443): Max retries exceeded with url: /realms/eoepca/protocol/openid-connect/auth/device (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)')))

In [ ]:
token = None
if owner_is_admin:
    token = owner_token
    creator = owner
else:
    admin_token, admin_is_admin = login_workspace_as("oscar")
    if not admin_is_admin:
        raise PermissionError("Oscar must have the workspace-api admin role. Check workspace-admin membership.")
    token = admin_token
    creator = "oscar"
print(f"{creator} will create the workspace for {owner}.")


Create a workspace for the selected owner. Its name includes a UTC timestamp.

If Oscar creates it, his token is also used for the storage steps below. The selected user remains `default_owner`.


In [ ]:
if not token:
    raise PermissionError("Complete the administrator check before creating a workspace.")
# The API limits the preferred name to 32 characters. Keep all 14 timestamp digits.
workspace_prefix = re.sub(r"[^a-z0-9]+", "-", owner.lower()).strip("-")[:17].rstrip("-")
if not workspace_prefix:
    raise ValueError("The username must contain a letter from a to z or a digit for the workspace name.")
timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d%H%M%S")
preferred_name = f"{workspace_prefix}-{timestamp}"
response = api.post(
    workspace_api_endpoint,
    headers={"Authorization": f"Bearer {token}"},
    json={"preferred_name": preferred_name, "default_owner": owner},
)
response.raise_for_status()
ws_name = response.json()["name"]
print(f"Workspace creation started: {ws_name} (owner: {owner})")

Wait for workspace storage to be ready before uploading files.


In [ ]:
workspace_data = wait_for_workspace(
    api, f"{workspace_api_endpoint}/{ws_name}",
    headers={"Authorization": f"Bearer {token}"},
)
print(f"Workspace ready: {ws_name}")

## Open the owner's workspaces

Sign in again as the owner to get a token with the newly assigned workspace roles. The cell lists the workspaces the owner can access and their page links.

For example, Alice can manage `ws-alice` and use `ws-bob`. Her access to Bob's workspace lets her open its dashboard and Datalab. Management links appear only for workspaces she can manage.

Open the links in a browser and sign in as the owner. Use the dashboard to start a session or the management page to edit members and bucket access.

The remaining upload and sharing cells use the creation token in `token`.


In [ ]:
from html import escape
from urllib.parse import quote

from IPython.display import HTML, display

owner_token, owner_is_admin = login_workspace_as(owner)
# These claims control which links are shown. The API checks permissions.
owner_claims = jwt.decode(owner_token, options={"verify_signature": False})
workspace_roles = owner_claims.get("resource_access", {})
response = api.get(
    workspace_api_endpoint,
    headers={"Authorization": f"Bearer {owner_token}", "Accept": "application/json"},
)
response.raise_for_status()
visible_workspaces = response.json()
if not visible_workspaces:
    print("No workspaces are available. Check the owner's workspace roles.")
for workspace in visible_workspaces:
    name = workspace["name"]
    roles = set(workspace_roles.get(name, {}).get("roles", []))
    url = f"{workspace_api_endpoint}/{quote(name, safe='')}"
    links = [("Dashboard", url)]
    if owner_is_admin or "ws_admin" in roles:
        links.append(("Management", f"{url}#/home/management"))
    for session in workspace.get("sessions", []):
        session_name = session["name"]
        links.append((
            f"Datalab ({session_name})",
            f"{url}/sessions/{quote(session_name, safe='')}",
        ))
    print(f"{name}: {', '.join(sorted(roles)) or 'no workspace role in token'}")
    display(HTML(" | ".join(
        f'<a href="{escape(href, quote=True)}">{escape(label)}</a>'
        for label, href in links
    )))


Upload `catalog.v1.parquet`, its rendered previews, and its thumbnails to workspace storage.


In [ ]:
import hashlib
import boto3
from botocore.config import Config
from IPython.display import HTML, Image, display
from html import escape
from notebook_support import (
    get_shared, preview_assets, public_file_url, require_https, verify_download,
)

credentials = workspace_data["storage"].get("credentials")
if not credentials:
    raise PermissionError("Workspace storage credentials are not available for this user.")
s3 = boto3.session.Session(
    aws_access_key_id=credentials["access"],
    aws_secret_access_key=credentials["secret"],
    region_name=credentials["region"],
).client(
    "s3", endpoint_url=require_https(credentials["endpoint"]),
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)
bucket = credentials["bucketname"]
public_bucket = ws_name
s3.upload_file(str(catalogue), bucket, catalogue.name,
               ExtraArgs={"ContentType": "application/vnd.apache.parquet"})
print(f"Uploaded {catalogue.name}: {len(items)} items")

# Use a separate session for external assets and shared downloads.
downloads = requests.Session()
downloads.trust_env = False

Fetch the catalogue's previews and thumbnails. Failed links are retried against the current STAC Item. Unreadable files are listed and skipped.


In [ ]:
uploaded, failed = [], []
for asset in preview_assets(items, downloads):
    if "error" in asset:
        failed.append(asset)
        print(f"Skipped {asset['item']} / {asset['asset']}: {asset['error']}")
        continue
    key = f"previews/{asset['filename']}"
    s3.put_object(Bucket=bucket, Key=key, Body=asset["content"],
                  ContentType=asset["type"])
    uploaded.append({
        "filename": asset["filename"], "key": key,
        "sha256": hashlib.sha256(asset["content"]).hexdigest(),
    })
    print(f"Uploaded {asset['filename']}")
print(f"Images uploaded: {len(uploaded)}; unavailable: {len(failed)}")
if not uploaded:
    raise RuntimeError("No preview images were available. Check the source assets before sharing.")

Display each uploaded image through a presigned link that expires after 15 minutes.


In [ ]:
for asset in uploaded:
    signed_url = s3.generate_presigned_url(
        "get_object", Params={"Bucket": bucket, "Key": asset["key"]},
        ExpiresIn=900,
    )
    shared = get_shared(downloads, signed_url)
    verify_download(shared, asset["sha256"])
    display(HTML(f'<a href="{escape(signed_url, quote=True)}">{escape(asset["filename"])}</a> valid for 15 mins'))
    display(Image(data=shared.content, width=360))

Copy the images into `public/` without subfolders. Anyone with the public link can read them.


In [ ]:
for asset in uploaded:
    s3.copy_object(
        Bucket=public_bucket, Key=f"public/{asset['filename']}",
        CopySource={"Bucket": bucket, "Key": asset["key"]},
    )
print(f"Copied {len(uploaded)} images to {public_bucket}/public/")

Enter the permanent Data URL from the workspace dashboard. Its default public share must be enabled.


In [ ]:
data_service_url = input(f"Permanent Data URL for {ws_name}: ").strip().rstrip("/")
for asset in uploaded:
    public_url = public_file_url(data_service_url, asset["filename"])
    shared = get_shared(downloads, public_url)
    if not any(hop.status_code in (301, 302, 303, 307, 308) for hop in shared.history):
        raise RuntimeError("The public endpoint did not return a download redirect.")
    verify_download(shared, asset["sha256"])
    chain = " → ".join(str(hop.status_code) for hop in [*shared.history, shared])
    display(HTML(f'<a href="{escape(public_url, quote=True)}">{escape(asset["filename"])}</a> ({chain})'))
    display(Image(data=shared.content, width=360))

Enter the exact workspace name to delete it, or leave the prompt empty to keep it. To confirm deletion, sign in again as the administrator who created it.


In [ ]:
confirm = input(f"Type {ws_name} to delete this workspace: ").strip()
if confirm == ws_name:
    delete_token, delete_is_admin = login_workspace_as(creator)
    if not delete_is_admin:
        raise PermissionError("The token must contain the workspace-api admin role to delete this workspace.")
    response = api.delete(
        f"{workspace_api_endpoint}/{ws_name}",
        headers={"Authorization": f"Bearer {delete_token}"},
    )
    response.raise_for_status()
    print(f"Workspace deletion started: {ws_name}")
else:
    print("Workspace kept.")